# Evaluating Natural Language Explanations of Food Recommendation System XFoodRec Across Several LLMs

Authors: Junxuan Ling, Tianyi Liu, Anqi Zhou

# 0 Document Instructions

## 0.1 Git Version Control

After making modifications, please use Colab's function of "Save in Github to keep changes" (Github authorization required) to commit the changes to the repository (Your own branch if needed).

**NOTE**: When A modifies this notebook, but have not commited to Github, if B opens this notebook, they will see the Github old version.

## 0.2 Handle persistent data storage

Mount Google Drive instead of relying on Colab's temporary run-time storage.

**IMPORTANT:** Although Google Colab allows collaboration of the notebook, data and folders are originally in Junxuan's drive, this could cause issues when refering to the mounted directory path. To allow the code to work for all of us, (who's not junxuan) please create a **short-cut** of Junxuan's shared folder in our own Google Drive, path `MyDrive/`.

In [417]:
# load from drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1 Data Exploration & Preparation

## 1.1 Load and Inspect

### a Load Data

In [418]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


data_dir = Path("/content/drive/MyDrive/XFoodRec_Project_Data/raw")
persona_path = data_dir / "personas.json"
recommendation_path = data_dir / "recommendations.json"
recipe_path = data_dir / "recipes.parquet"

In [419]:
# open files and load data

with open(persona_path, "r", encoding="utf-8") as f:
    personas = json.load(f)

with open(recommendation_path, "r", encoding="utf-8") as f:
    recommendations = json.load(f)

recipes = pd.read_parquet(recipe_path)

### b Inspect Raw data

In [420]:
import json

In [421]:
print("personas")
print("type:", type(personas))
print("length:", len(personas))

print(json.dumps(personas[0],indent=2))

personas
type: <class 'list'>
length: 40
{
  "id": "user_01",
  "description": "A vegan competitive bodybuilder focused on lean bulking without soy overload.",
  "profile": {
    "age": 28,
    "gender": "Male",
    "height": 182.0,
    "weight": 84.0,
    "activityLevel": "very_active",
    "dietary_goal": "Muscle Gain",
    "dietaryProfile": {
      "dietaryRestrictions": {
        "selected": [
          "Vegan"
        ],
        "other": "High-protein, prefers low-soy options"
      },
      "foodAllergies": {
        "selected": [],
        "other": ""
      },
      "healthConditions": {
        "selected": [],
        "other": ""
      }
    },
    "likedIngredients": [
      "Seitan",
      "Lentils",
      "Chickpeas",
      "Hemp seeds"
    ],
    "dislikedIngredients": [
      "Mushrooms",
      "Eggplant"
    ],
    "favoriteCuisines": [
      "Middle Eastern",
      "Indian",
      "Mediterranean"
    ]
  }
}


In [422]:
print("\nrecommendations")
print("type:", type(recommendations))
print("length:", len(recommendations))

print(json.dumps(recommendations[0],indent=2))


recommendations
type: <class 'list'>
length: 40
{
  "persona": {
    "id": "user_01",
    "description": "A vegan competitive bodybuilder focused on lean bulking without soy overload.",
    "profile": {
      "age": 28,
      "gender": "Male",
      "height": 182.0,
      "weight": 84.0,
      "activityLevel": "very_active",
      "dietary_goal": "Muscle Gain",
      "dietaryProfile": {
        "dietaryRestrictions": {
          "selected": [
            "Vegan"
          ],
          "other": "High-protein, prefers low-soy options"
        },
        "foodAllergies": {
          "selected": [],
          "other": ""
        },
        "healthConditions": {
          "selected": [],
          "other": ""
        }
      },
      "likedIngredients": [
        "Seitan",
        "Lentils",
        "Chickpeas",
        "Hemp seeds"
      ],
      "dislikedIngredients": [
        "Mushrooms",
        "Eggplant"
      ],
      "favoriteCuisines": [
        "Middle Eastern",
        "Indian"

In [423]:
print("\nrecipes")
print("type:", type(recipes))
print("shape:", recipes.shape)

display(recipes.head())



recipes
type: <class 'pandas.core.frame.DataFrame'>
shape: (702, 31)


,recipe_id,title,recipe_url,image_url,ingredients,ingredients_title,tags,calories_per_serving [cal],calories_per_100g [cal],caloriesfromfat_per_serving [cal],caloriesfromfat_per_100g [cal],totalfat_per_serving [g],totalfat_per_100g [g],saturatedfat_per_serving [g],saturatedfat_per_100g [g],cholesterol_per_serving [mg],cholesterol_per_100g [mg],sodium_per_serving [mg],sodium_per_100g [mg],totalcarbohydrate_per_serving [g],totalcarbohydrate_per_100g [g],dietaryfiber_per_serving [g],dietaryfiber_per_100g [g],sugars_per_serving [g],sugars_per_100g [g],protein_per_serving [g],protein_per_100g [g],who_score,fsa_score,nutri_score,health_score
0,113861,My Own Cube Steak and Gravy,https://www.food.com/recipe/my-own-cube-steak-...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('cubed steak (4-6 pieces)', '2 time(s)...","[cubed steak (4-6 pieces), oil, butter, flour,...","[Moderate Calorie, Vegetarian]",500.5,175.6,216,75.8,24.0,8.4,9.4,3.3,147.5,51.8,824.1,289.2,20.6,7.2,0.8,0.3,0.8,0.3,48.0,16.8,0.266834,0.250,0.00,3.6
1,111212,Our Secret Sirloin Steak,https://www.food.com/recipe/our-secret-sirloin...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('Lea & Perrins Worcestershire Sauce, pl...","[Lea & Perrins Worcestershire Sauce, Lea & Per...","[Moderate Calorie, Vegetarian]",823.6,320.5,502,195.3,55.8,21.7,22.0,8.6,270.4,105.2,847.5,329.8,7.2,2.8,0.4,0.2,3.4,1.3,69.5,27.0,0.186863,0.250,0.00,2.9
2,497021,Mile-High Cabbage Pie #5FIX,https://www.food.com/recipe/mile-high-cabbage-...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('lean ground beef', '1 1/2 time(s) lbs ...","[lean beef, coleslaw mix (cabbage and carrots)...",[Moderate Calorie],803.0,221.2,298,82.1,33.2,9.1,10.8,3.0,167.4,46.1,1698.9,468.0,75.7,20.9,7.0,1.9,10.7,2.9,48.7,13.4,0.266693,0.250,0.00,3.6
3,31184,15 Minute Shrimp Scampi,https://www.food.com/recipe/15-minute-shrimp-s...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('extra virgin olive oil', '0.5 time(s) ...","[extra virgin olive oil, onion, cloves garlic,...",[Moderate Calorie],596.2,342.6,416,239.1,46.3,26.6,14.9,8.6,284.9,163.7,1231.1,707.5,8.2,4.7,0.7,0.4,2.2,1.3,26.7,15.3,0.169814,0.250,0.00,2.7
4,71739,Easy Garlic Rice,https://www.food.com/recipe/easy-garlic-rice-7...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('olive oil', '2 time(s) tablespoons ')...","[olive oil, onion, garlic cloves, long grain r...",[Moderate Calorie],264.2,190.1,70,50.4,7.8,5.6,1.2,0.9,0.0,0.0,379.4,272.9,41.1,29.6,1.3,0.9,1.6,1.2,6.3,4.5,0.222165,0.625,0.25,5.2


### c Organize Raw Data as DataFrames and Further Inspect

In [424]:
# pd display setting
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

#### c.1 persona

In [425]:
personas_df = pd.json_normalize(personas)

print(personas_df.shape)
display(personas_df.head())

(40, 17)


,id,description,profile.age,profile.gender,profile.height,profile.weight,profile.activityLevel,profile.dietary_goal,profile.dietaryProfile.dietaryRestrictions.selected,profile.dietaryProfile.dietaryRestrictions.other,profile.dietaryProfile.foodAllergies.selected,profile.dietaryProfile.foodAllergies.other,profile.dietaryProfile.healthConditions.selected,profile.dietaryProfile.healthConditions.other,profile.likedIngredients,profile.dislikedIngredients,profile.favoriteCuisines
0,user_01,A vegan competitive bodybuilder focused on lea...,28,Male,182.0,84.0,very_active,Muscle Gain,[Vegan],"High-protein, prefers low-soy options",[],,[],,"[Seitan, Lentils, Chickpeas, Hemp seeds]","[Mushrooms, Eggplant]","[Middle Eastern, Indian, Mediterranean]"
1,user_02,A type 1 diabetic with a severe tree-nut aller...,34,Female,168.0,67.0,moderately_active,Medical Management,[Low Carb],Counts carbs precisely; prefers low-GI foods,[Tree Nuts],Avoids cross-contamination,[Diabetes (Type 1)],,"[Chicken, Greek yogurt, Broccoli, Berries]","[Sweetened cereals, Fruit juice]","[American, Mediterranean]"
2,user_03,A low-budget student with lactose intolerance ...,20,Non-binary,172.0,92.0,lightly_active,Weight Loss,[Lactose-Free],Very budget-constrained; batch-cooks,[],,[],,"[Rice, Beans, Frozen vegetables, Canned tuna]","[Cottage cheese, Cream sauces]","[Mexican, Asian]"
3,user_04,A pregnant vegetarian with iron-deficiency ane...,31,Female,160.0,64.0,lightly_active,Medical Management,[Vegetarian],"Pregnant; prioritizes iron, folate, choline",[],,[Anemia],Iron-deficiency anemia,"[Eggs, Lentils, Spinach, Fortified cereal]","[Tofu, Beets]","[Mediterranean, Indian]"
4,user_05,A marathon runner with celiac disease optimizi...,26,Male,177.0,66.0,very_active,Energy Boost,[Gluten-Free],Endurance fueling; needs portable snacks,[],,[Celiac Disease],,"[Rice, Potatoes, Bananas, Oats (certified GF)]","[Wheat bread, Barley]","[Japanese, Mexican]"


In [426]:
personas_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 17 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   id                                                   40 non-null     object 
 1   description                                          40 non-null     object 
 2   profile.age                                          40 non-null     int64  
 3   profile.gender                                       40 non-null     object 
 4   profile.height                                       40 non-null     float64
 5   profile.weight                                       40 non-null     float64
 6   profile.activityLevel                                40 non-null     object 
 7   profile.dietary_goal                                 40 non-null     object 
 8   profile.dietaryProfile.dietaryRestrictions.selected  40 non-null     obj

In [427]:
personas_df.isna().sum()

,0
id,0
description,0
profile.age,0
profile.gender,0
profile.height,0
profile.weight,0
profile.activityLevel,0
profile.dietary_goal,0
profile.dietaryProfile.dietaryRestrictions.selected,0
profile.dietaryProfile.dietaryRestrictions.other,0


In [428]:
print("unique personas:", personas_df["id"].nunique())
print("duplicate ids:", personas_df["id"].duplicated().sum())

unique personas: 40
duplicate ids: 0


In [429]:
# persona distribution

distribution_columns = {
    "Gender": "profile.gender",
    "Dietary Goal": "profile.dietary_goal",
    "Activity Level": "profile.activityLevel",
    "Dietary Restriction": "profile.dietaryProfile.dietaryRestrictions.selected",
    "Food Allergy": "profile.dietaryProfile.foodAllergies.selected",
    "Health Condition": "profile.dietaryProfile.healthConditions.selected"
}

persona_distribution = {}

for name, column in distribution_columns.items():
    values = personas_df[column]

    if values.apply(lambda x: isinstance(x, list)).all():
        values = values.explode()

    counts = values.value_counts()
    persona_distribution[name] = pd.Series(
        [f"{category}: {count}" for category, count in counts.items()]
    )

display(pd.DataFrame(persona_distribution).fillna(""))

,Gender,Dietary Goal,Activity Level,Dietary Restriction,Food Allergy,Health Condition
0,Female: 20,Medical Management: 14,lightly_active: 14,High Protein: 3,Tree Nuts: 2,Food Allergy: 2
1,Male: 19,Maintenance: 8,moderately_active: 13,Vegan: 2,Peanuts: 2,Diabetes (Type 1): 1
2,Non-binary: 1,Weight Loss: 7,sedentary: 7,Keto: 2,Milk: 2,Celiac Disease: 1
3,,Energy Boost: 6,very_active: 6,Lactose-Free: 2,Eggs: 2,Post-Surgery: 1
4,,Muscle Gain: 5,,Low Fiber: 2,Shellfish: 1,High Cholesterol: 1
5,,,,Low Sodium: 2,Wheat: 1,Anemia: 1
6,,,,Anti-Inflammatory: 2,Soy: 1,Kidney Disease: 1
7,,,,Low Fat: 2,Sesame: 1,GERD: 1
8,,,,Low Carb: 1,Apples: 1,IBS: 1
9,,,,Vegetarian: 1,,Autism Spectrum: 1


#### c.2 recommendation

In [430]:
# persona -> recommendations

recommendation_rows = []

for item in recommendations:
    persona_id = item["persona"]["id"]

    for rec in item["recommendations"]:
        recommendation_rows.append({
            "persona_id": persona_id,
            **rec
        })

recommendations_df = pd.DataFrame(recommendation_rows)

print(recommendations_df.shape)
display(recommendations_df.head())

(536, 6)


,persona_id,recipe_id,title,explanation,ingredients,nutrition
0,user_01,81968,Chickpeas With Spinach (Greek),Chickpeas With Spinach (Greek) is a fantastic ...,"[onions, garlic cloves, olive oil, cooked chic...","{'calories': 310.2, 'protein': '13.3g', 'carbs..."
1,user_01,10614,Refried Beans,"Refried Beans offers a high protein content, m...","[oil, jalapenos, onions, (optional), black bea...","{'calories': 307.1, 'protein': '17.6g', 'carbs..."
2,user_01,94673,Egyptian Red Lentil Soup,Egyptian Red Lentil Soup is rich in protein an...,"[vegetable broth, red lentils, onions, potatoe...","{'calories': 204.5, 'protein': '10.3g', 'carbs..."
3,user_01,364252,My Favorite Sauteed Kale,My Favorite Sauteed Kale is a nutrient-dense o...,"[kale, olive oil, red pepper flakes, garlic cl...","{'calories': 154.2, 'protein': '5.8g', 'carbs'..."
4,user_01,12569,Hearty Black Beans & Rice,Hearty Black Beans & Rice combines protein-ric...,"[black beans (and liquid), onion, cloves garli...","{'calories': 533.8, 'protein': '11.4g', 'carbs..."


In [431]:
recommendations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 536 entries, 0 to 535
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   persona_id   536 non-null    object
 1   recipe_id    536 non-null    object
 2   title        536 non-null    object
 3   explanation  536 non-null    object
 4   ingredients  536 non-null    object
 5   nutrition    536 non-null    object
dtypes: object(6)
memory usage: 25.3+ KB


In [432]:
print(recommendations_df.columns.tolist())

['persona_id', 'recipe_id', 'title', 'explanation', 'ingredients', 'nutrition']


In [433]:
personas_df["profile.dietaryProfile.dietaryRestrictions.selected"].explode().value_counts(dropna=False)

,count
profile.dietaryProfile.dietaryRestrictions.selected,
NaN,8
High Protein,3
Vegan,2
Keto,2
Lactose-Free,2
Low Fiber,2
Low Sodium,2
Anti-Inflammatory,2
Low Fat,2


In [434]:
# recommentation number for personas

rec_counts = recommendations_df.groupby("persona_id").size()

display(rec_counts.describe())

,0
count,40.000000
mean,13.400000
std,1.335895
min,10.000000
25%,12.750000
50%,13.500000
75%,14.000000
max,16.000000


In [435]:
display(rec_counts.sort_values())

,0
persona_id,
user_12,10
user_02,11
user_36,11
user_20,12
user_30,12
user_29,12
user_34,12
user_17,12
user_37,12


In [436]:
print("total recommendations:", len(recommendations_df))
print("personas:", recommendations_df["persona_id"].nunique())
print("unique recipes:", recommendations_df["recipe_id"].nunique())

total recommendations: 536
personas: 40
unique recipes: 173


In [437]:
duplicate_recs = recommendations_df.duplicated(
    subset=["persona_id", "recipe_id"],
    keep=False
)

print("duplicate recommendation rows:", duplicate_recs.sum())

duplicate recommendation rows: 89


In [438]:
display(
    recommendations_df.loc[
        duplicate_recs,
        ["persona_id", "recipe_id", "title", "explanation"]
    ].sort_values(["persona_id", "recipe_id"])
)

,persona_id,recipe_id,title,explanation
5,user_01,27778,Black Bean and Corn Salad,"Black Bean and Corn Salad is a colorful, nutri..."
7,user_01,27778,Black Bean and Corn Salad,"Black Bean and Corn Salad is a colorful, nutri..."
3,user_01,364252,My Favorite Sauteed Kale,My Favorite Sauteed Kale is a nutrient-dense o...
13,user_01,364252,My Favorite Sauteed Kale,My Favorite Sauteed Kale is a nutrient-dense o...
9,user_01,46221,Classic Black Beans and Rice,Classic Black Beans and Rice is a traditional ...
...,...,...,...,...
431,user_32,8594,Fried Rice,Fried Rice is a versatile dish that can be enj...
460,user_35,30391,Chicken Souvlaki,Chicken Souvlaki is a flavorful option that al...
466,user_35,30391,Chicken Souvlaki,Chicken Souvlaki is a flavorful option that al...
515,user_39,15344,Easy Weeknight Sloppy Joes,Easy Weeknight Sloppy Joes are a hearty option...


In [439]:
print(
    "unique persona-recipe pairs:",
    recommendations_df[
        ["persona_id", "recipe_id"]
    ].drop_duplicates().shape[0]
)

unique persona-recipe pairs: 484


#### c.3 recipe

In [440]:
print(recommendations_df["recipe_id"].dtype)
print(recipes["recipe_id"].dtype)

object
int64


In [441]:
# join 2 df

recommendations_df["recipe_id"] = recommendations_df["recipe_id"].astype(str)
recipes["recipe_id"] = recipes["recipe_id"].astype(str)

matched = recommendations_df["recipe_id"].isin(recipes["recipe_id"])

print("matched:", matched.sum())
print("unmatched:", (~matched).sum())
print("match rate:", matched.mean())

matched: 536
unmatched: 0
match rate: 1.0


In [442]:
# extract nutrition part and attach it to the larger df

nutrition_df = pd.json_normalize(recommendations_df["nutrition"])

display(nutrition_df.head())

,calories,protein,carbs,fat
0,310.2,13.3g,54.2g,6.2g
1,307.1,17.6g,51.1g,2.6g
2,204.5,10.3g,35.2g,3.0g
3,154.2,5.8g,18.9g,8.0g
4,533.8,11.4g,101.5g,11.2g


In [443]:
recommendations_df = pd.concat(
    [
        recommendations_df.drop(columns="nutrition"),
        nutrition_df.add_prefix("rec_")
    ],
    axis=1
)

In [444]:
# merge recommendation and recipe

merged_df = recommendations_df.merge(
    recipes,
    on="recipe_id",
    how="left",
    suffixes=("_rec", "_recipe")
)

print(merged_df.shape)
display(merged_df.head())

(536, 39)


,persona_id,recipe_id,title_rec,explanation,ingredients_rec,rec_calories,rec_protein,rec_carbs,rec_fat,title_recipe,recipe_url,image_url,ingredients_recipe,ingredients_title,tags,calories_per_serving [cal],calories_per_100g [cal],caloriesfromfat_per_serving [cal],caloriesfromfat_per_100g [cal],totalfat_per_serving [g],totalfat_per_100g [g],saturatedfat_per_serving [g],saturatedfat_per_100g [g],cholesterol_per_serving [mg],cholesterol_per_100g [mg],sodium_per_serving [mg],sodium_per_100g [mg],totalcarbohydrate_per_serving [g],totalcarbohydrate_per_100g [g],dietaryfiber_per_serving [g],dietaryfiber_per_100g [g],sugars_per_serving [g],sugars_per_100g [g],protein_per_serving [g],protein_per_100g [g],who_score,fsa_score,nutri_score,health_score
0,user_01,81968,Chickpeas With Spinach (Greek),Chickpeas With Spinach (Greek) is a fantastic ...,"[onions, garlic cloves, olive oil, cooked chic...",310.2,13.3g,54.2g,6.2g,Chickpeas With Spinach (Greek),https://www.food.com/recipe/chickpeas-with-spi...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('medium onions, chopped', '2 time(s) '...","[onions, garlic cloves, olive oil, cooked chic...","[Moderate Calorie, Vegan, Vegetarian]",310.2,76.6,55,13.6,6.2,1.5,0.8,0.2,0.0,0.0,602.7,148.8,54.2,13.4,12.6,3.1,5.1,1.3,13.3,3.3,0.275309,0.75,0.50,6.9
1,user_01,10614,Refried Beans,"Refried Beans offers a high protein content, m...","[oil, jalapenos, onions, (optional), black bea...",307.1,17.6g,51.1g,2.6g,Refried Beans,https://www.food.com/recipe/refried-beans-10614,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('oil', '2 time(s) teaspoons '), ('jala...","[oil, jalapenos, onions, (optional), black bea...","[Moderate Calorie, Vegan, Vegetarian]",307.1,139.0,23,10.4,2.6,1.2,0.5,0.2,0.0,0.0,5.9,2.7,51.1,23.1,17.4,7.9,1.8,0.8,17.6,8.0,0.349709,1.00,1.00,10.0
2,user_01,94673,Egyptian Red Lentil Soup,Egyptian Red Lentil Soup is rich in protein an...,"[vegetable broth, red lentils, onions, potatoe...",204.5,10.3g,35.2g,3.0g,Egyptian Red Lentil Soup,https://www.food.com/recipe/egyptian-red-lenti...,https://img.sndimg.com/food/image/upload/f_aut...,{'': [('vegetable broth or 5 cups $template2$'...,"[vegetable broth, red lentils, onions, potatoe...","[Moderate Calorie, Vegan, Vegetarian]",204.5,181.0,26,23.0,3.0,2.7,0.3,0.3,0.0,0.0,397.0,351.3,35.2,31.2,12.0,10.6,3.6,3.2,10.3,9.1,0.261062,0.75,0.50,6.8
3,user_01,364252,My Favorite Sauteed Kale,My Favorite Sauteed Kale is a nutrient-dense o...,"[kale, olive oil, red pepper flakes, garlic cl...",154.2,5.8g,18.9g,8.0g,My Favorite Sauteed Kale,https://www.food.com/recipe/my-favorite-sautee...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('kale, stems and leaves coarsely choppe...","[kale, olive oil, red pepper flakes, garlic cl...","[Moderate Calorie, Vegan, Vegetarian]",154.2,82.9,71,38.2,8.0,4.3,1.1,0.6,0.0,0.0,75.6,40.6,18.9,10.2,3.4,1.8,1.2,0.6,5.8,3.1,0.320405,0.75,0.50,7.4
4,user_01,12569,Hearty Black Beans & Rice,Hearty Black Beans & Rice combines protein-ric...,"[black beans (and liquid), onion, cloves garli...",533.8,11.4g,101.5g,11.2g,Hearty Black Beans & Rice,https://www.food.com/recipe/black-beans-and-ri...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('black beans (and liquid)', '1 time(s)...","[black beans (and liquid), onion, cloves garli...","[Moderate Calorie, Vegan, Vegetarian]",533.8,104.9,100,19.6,11.2,2.2,1.7,0.3,0.0,0.0,434.8,85.4,101.5,19.9,14.1,2.8,8.3,1.6,11.4,2.2,0.266208,0.75,0.25,6.1


In [445]:
# check if all matched successfully

print(
    "recommendations without recipe data:",
    merged_df["title_recipe"].isna().sum()
)

recommendations without recipe data: 0


In [446]:
# dlt duplicated columns

merged_df = merged_df.drop(columns="title_recipe")
merged_df = merged_df.drop(columns="ingredients_title")
merged_df = merged_df.rename(columns={"title_rec": "title"})
merged_df = merged_df.rename(columns={"ingredients_rec": "ingredients"})
merged_df = merged_df.rename(columns={"rec_calories": "calories"})
merged_df = merged_df.rename(columns={"rec_protein": "protein"})
merged_df = merged_df.rename(columns={"rec_carbs": "carbs"})
merged_df = merged_df.rename(columns={"rec_fat": "fat"})
merged_df = merged_df.rename(columns={"ingredients_recipe": "ingredients_detailed"})

print(merged_df.shape)
display(merged_df.head())

(536, 37)


,persona_id,recipe_id,title,explanation,ingredients,calories,protein,carbs,fat,recipe_url,image_url,ingredients_detailed,tags,calories_per_serving [cal],calories_per_100g [cal],caloriesfromfat_per_serving [cal],caloriesfromfat_per_100g [cal],totalfat_per_serving [g],totalfat_per_100g [g],saturatedfat_per_serving [g],saturatedfat_per_100g [g],cholesterol_per_serving [mg],cholesterol_per_100g [mg],sodium_per_serving [mg],sodium_per_100g [mg],totalcarbohydrate_per_serving [g],totalcarbohydrate_per_100g [g],dietaryfiber_per_serving [g],dietaryfiber_per_100g [g],sugars_per_serving [g],sugars_per_100g [g],protein_per_serving [g],protein_per_100g [g],who_score,fsa_score,nutri_score,health_score
0,user_01,81968,Chickpeas With Spinach (Greek),Chickpeas With Spinach (Greek) is a fantastic ...,"[onions, garlic cloves, olive oil, cooked chic...",310.2,13.3g,54.2g,6.2g,https://www.food.com/recipe/chickpeas-with-spi...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('medium onions, chopped', '2 time(s) '...","[Moderate Calorie, Vegan, Vegetarian]",310.2,76.6,55,13.6,6.2,1.5,0.8,0.2,0.0,0.0,602.7,148.8,54.2,13.4,12.6,3.1,5.1,1.3,13.3,3.3,0.275309,0.75,0.50,6.9
1,user_01,10614,Refried Beans,"Refried Beans offers a high protein content, m...","[oil, jalapenos, onions, (optional), black bea...",307.1,17.6g,51.1g,2.6g,https://www.food.com/recipe/refried-beans-10614,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('oil', '2 time(s) teaspoons '), ('jala...","[Moderate Calorie, Vegan, Vegetarian]",307.1,139.0,23,10.4,2.6,1.2,0.5,0.2,0.0,0.0,5.9,2.7,51.1,23.1,17.4,7.9,1.8,0.8,17.6,8.0,0.349709,1.00,1.00,10.0
2,user_01,94673,Egyptian Red Lentil Soup,Egyptian Red Lentil Soup is rich in protein an...,"[vegetable broth, red lentils, onions, potatoe...",204.5,10.3g,35.2g,3.0g,https://www.food.com/recipe/egyptian-red-lenti...,https://img.sndimg.com/food/image/upload/f_aut...,{'': [('vegetable broth or 5 cups $template2$'...,"[Moderate Calorie, Vegan, Vegetarian]",204.5,181.0,26,23.0,3.0,2.7,0.3,0.3,0.0,0.0,397.0,351.3,35.2,31.2,12.0,10.6,3.6,3.2,10.3,9.1,0.261062,0.75,0.50,6.8
3,user_01,364252,My Favorite Sauteed Kale,My Favorite Sauteed Kale is a nutrient-dense o...,"[kale, olive oil, red pepper flakes, garlic cl...",154.2,5.8g,18.9g,8.0g,https://www.food.com/recipe/my-favorite-sautee...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('kale, stems and leaves coarsely choppe...","[Moderate Calorie, Vegan, Vegetarian]",154.2,82.9,71,38.2,8.0,4.3,1.1,0.6,0.0,0.0,75.6,40.6,18.9,10.2,3.4,1.8,1.2,0.6,5.8,3.1,0.320405,0.75,0.50,7.4
4,user_01,12569,Hearty Black Beans & Rice,Hearty Black Beans & Rice combines protein-ric...,"[black beans (and liquid), onion, cloves garli...",533.8,11.4g,101.5g,11.2g,https://www.food.com/recipe/black-beans-and-ri...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('black beans (and liquid)', '1 time(s)...","[Moderate Calorie, Vegan, Vegetarian]",533.8,104.9,100,19.6,11.2,2.2,1.7,0.3,0.0,0.0,434.8,85.4,101.5,19.9,14.1,2.8,8.3,1.6,11.4,2.2,0.266208,0.75,0.25,6.1


## 1.2 Extract all explanations and compute basic statistics